# SPACY et Universal Dependencies

Illustration de Spacy, exemples tiré de Thérèse Desqueyroux (Mauriac)

https://ia601700.us.archive.org/10/items/isbn_9789200325229/isbn_9789200325229.pdf


In [24]:
# doc spacy
# https://spacy.io/usage/linguistic-features

import spacy
from spacy import displacy
nlp = spacy.load('fr_core_news_md')
import pandas as pd

In [34]:
myTxt = "Il se rendit au parc."

In [35]:
doc = nlp(myTxt)

In [36]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 100})

 mod (pas ds la structure du verbe, ajout => sert de cadre, portée plus grande que la phrase), arg (complément indirect), agent (complément d'agent)
acl (complément du nom, ex: "le bar d'en haut")

mod => ajouter si c'est avant le sujet + le sujet est avant le verbe => cadre du discours
apres le verbe => localisation locale (pas cadre) => en general, il faut modeliser pour voir aussi si c'est bon

d'abord filtrer par verbe dynamouv ou pas

prepostion en case => entrainer pour chaque preposition avec son contexte a quel point c'est un indicateur de but, source, direction ou le median.

separation de chapitre

expl:comp => les pronominaux se retrouvent en objet

des moments ou c'est statique et donc ou on ne trouve pas => si un modele contraint ou ca se passe ca peut poser probleme, il faut marquer un implicite ou meme un flou => une sorte de certitude de localisation

In [5]:
for token in doc:
    print("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}\t{8}".format(
        token.text,
        token.i,
        token.lemma_,
        token.is_punct,
        token.is_space,
        token.shape_,
        token.pos_,
        token.dep_,
        token.head.i
    ))

La	0	le	False	False	Xx	DET	det	1
voiture	1	voiture	False	False	xxxx	NOUN	ROOT	1
part	2	part	False	False	xxxx	NOUN	amod	1
pour	3	pour	False	False	xxxx	ADP	case	4
Douvres	4	Douvres	False	False	Xxxxx	PROPN	nmod	1
et	5	et	False	False	xx	CCONJ	cc	6
Calais	6	Calais	False	False	Xxxxx	PROPN	conj	4


In [6]:
# donne tous les paterns nominaux
doc.noun_chunks

In [7]:
for chunk in doc.noun_chunks:
    print(f"{chunk.text} / {chunk.root.text} / {chunk.root.dep_} / {chunk.root.head.text}")


La voiture part / voiture / ROOT / voiture
Douvres / Douvres / nmod / voiture
Calais / Calais / conj / Douvres


In [8]:
for token in doc:
    print(token.text, token.dep_, token.head.text, token.head.pos_,
            [child for child in token.children])

La det voiture NOUN []
voiture ROOT voiture NOUN [La, part, Douvres]
part amod voiture NOUN []
pour case Douvres PROPN []
Douvres nmod voiture NOUN [pour, Calais]
et cc Calais PROPN []
Calais conj Douvres PROPN [et]


In [9]:
root = [token for token in doc if token.head == token][0]
root.lefts

subject = list(root.lefts)[0]

for descendant in subject.subtree:
    assert subject is descendant or subject.is_ancestor(descendant)
    print(descendant.text, descendant.dep_, descendant.n_lefts,
            descendant.n_rights,
            [ancestor.text for ancestor in descendant.ancestors])

La det 0 0 ['voiture']


In [10]:
# Ajout d'un modifieur (adj) à droite et à gauche du nom 'avocat'
myTxt = "Le grand avocat français ouvrit une porte."

doc = nlp(myTxt)

root = [token for token in doc if token.head == token][0]
root.lefts

# on prend l'élem à gauche du root 
subject = list(root.lefts)[0]

# pour cet élement (donc ici le sujet) 
# ...on ragerde si il a des descendant à droite et à gauche
for descendant in subject.subtree:
    print(descendant.text, descendant.dep_, descendant.n_lefts,
            descendant.n_rights,
            [ancestor.text for ancestor in descendant.ancestors])

Le det 0 0 ['avocat', 'ouvrit']
grand amod 0 0 ['avocat', 'ouvrit']
avocat nsubj 2 1 ['ouvrit']
français amod 0 0 ['avocat', 'ouvrit']


In [11]:
displacy.render(doc, style='dep', jupyter=True, options={'distance': 100})

In [12]:
# on prend l'élem à droite du root 
myObject = list(root.rights)[0]

# pour cet élement (donc ici le complément) 
# ...on ragerde si il a des descendant à droite et à gauche
for descendant in myObject.subtree:
    print(descendant.text, descendant.dep_, descendant.n_lefts,
            descendant.n_rights,
            [ancestor.text for ancestor in descendant.ancestors])

une det 0 0 ['porte', 'ouvrit']
porte obj 1 0 ['ouvrit']


In [13]:
# parcourir l'arbre
for token in doc :
    print("-----")
    print("TOKEN :: "+str(token.text))
    print("TETE (token) :: "+str(token.head.text))
    print(token.text, token.dep_, token.head.text, token.head.pos_,
            [child for child in token.children])

-----
TOKEN :: Le
TETE (token) :: avocat
Le det avocat NOUN []
-----
TOKEN :: grand
TETE (token) :: avocat
grand amod avocat NOUN []
-----
TOKEN :: avocat
TETE (token) :: ouvrit
avocat nsubj ouvrit VERB [Le, grand, français]
-----
TOKEN :: français
TETE (token) :: avocat
français amod avocat NOUN []
-----
TOKEN :: ouvrit
TETE (token) :: ouvrit
ouvrit ROOT ouvrit VERB [avocat, porte, .]
-----
TOKEN :: une
TETE (token) :: porte
une det porte NOUN []
-----
TOKEN :: porte
TETE (token) :: ouvrit
porte obj ouvrit VERB [une]
-----
TOKEN :: .
TETE (token) :: ouvrit
. punct ouvrit VERB []


In [14]:
for token in doc:
    print("{0}\t{1}\t{2}\t{3}\t{4}\t{5}\t{6}\t{7}\t{8}".format(
        token.text,
        token.idx,
        token.lemma_,
        token.is_punct,
        token.is_space,
        token.shape_,
        token.pos_,
        token.tag_,
        token.ent_type_
    ))

Le	0	le	False	False	Xx	DET	DET	
grand	3	grand	False	False	xxxx	ADJ	ADJ	
avocat	9	avocat	False	False	xxxx	NOUN	NOUN	
français	16	français	False	False	xxxx	ADJ	ADJ	
ouvrit	25	ouvrir	False	False	xxxx	VERB	VERB	
une	32	un	False	False	xxx	DET	DET	
porte	36	porte	False	False	xxxx	NOUN	NOUN	
.	41	.	True	False	.	PUNCT	PUNCT	


In [15]:
myDebRoman = "L’avocat ouvrit une porte. Thérése Desqueyroux, dans ce couloir dérobé du Palais de justice, sentit sur sa face la brume et profondément, l’aspira. Elle avait peur d'être attendue, hésitait a sortir. Un homme, dont le col était relevé, se détacha d’un platane ; elle reconnut son père. "
docDebRoman = nlp(myDebRoman)


In [16]:
for sent in docDebRoman.sents:
    print(sent)

L’avocat ouvrit une porte.
Thérése Desqueyroux, dans ce couloir dérobé du Palais de justice, sentit sur sa face la brume et profondément, l’aspira.
Elle avait peur d'être attendue, hésitait a sortir.
Un homme, dont le col était relevé, se détacha d’un platane ;
elle reconnut son père.


In [17]:
for chunk in docDebRoman.noun_chunks:
    print(chunk.text, " --> ", chunk.label_)

L’avocat  -->  NP
une porte  -->  NP
Thérése Desqueyroux, dans ce couloir dérobé du Palais de justice, sentit sur sa face la brume et profondément, l’aspira.  -->  NP
Elle  -->  NP
peur  -->  NP
Un homme  -->  NP
dont  -->  NP
un platane  -->  NP
elle  -->  NP
son père  -->  NP


In [18]:
for ent in docDebRoman.ents:
    print(ent.text, ent.label_)

Thérése Desqueyroux PER
Palais de justice LOC


In [19]:
import stanza

stanza.download("fr")   # à faire une seule fois
nlp = stanza.Pipeline("fr")

doc = nlp("Jean part quand Marie arrive.")

for sent in doc.sentences:
    for w in sent.words:
        print(
            w.text,
            w.upos,
            w.deprel,
            "→ head:", w.head
        )


2026-02-23 15:56:15 INFO: Downloaded file to C:\Users\arnod\stanza_resources\resources.json
2026-02-23 15:56:15 INFO: Downloading default packages for language: fr (French) ...
2026-02-23 15:56:17 INFO: File exists: C:\Users\arnod\stanza_resources\fr\default.zip
2026-02-23 15:56:21 INFO: Finished downloading models and saved to C:\Users\arnod\stanza_resources
2026-02-23 15:56:21 INFO: Checking for updates to resources.json in case models have been updated.  Note: this behavior can be turned off with download_method=None or download_method=DownloadMethod.REUSE_RESOURCES
2026-02-23 15:56:22 INFO: Downloaded file to C:\Users\arnod\stanza_resources\resources.json
2026-02-23 15:56:24 INFO: Loading these models for language: fr (French):
| Processor | Package            |
----------------------------------
| tokenize  | combined           |
| mwt       | combined           |
| pos       | combined_charlm    |
| lemma     | combined_nocharlm  |
| depparse  | combined_charlm    |
| ner       |

Jean PROPN nsubj → head: 2
part VERB root → head: 0
quand SCONJ mark → head: 5
Marie PROPN nsubj → head: 5
arrive VERB advcl → head: 2
. PUNCT punct → head: 2


In [20]:
myTxt = "Au milieu de la rue Saint-Denis, presque au coin de la rue du Petit-Lion, existait naguère une de ces maisons précieuses qui donnent aux historiens la facilité de reconstruire par analogie l' ancien Paris."
doc = nlp(myTxt)

for sent in doc.sentences:
    for w in sent.words:
        print(
            w.id,
            w.text,
            w.upos,
            w.deprel,
            "→ head:", w.head
        )


1 à ADP case → head: 3
2 le DET det → head: 3
3 milieu NOUN obl:mod → head: 20
4 de ADP case → head: 6
5 la DET det → head: 6
6 rue NOUN nmod → head: 3
7 Saint-Denis PROPN appos → head: 6
8 , PUNCT punct → head: 12
9 presque ADV advmod → head: 12
10 à ADP case → head: 12
11 le DET det → head: 12
12 coin NOUN nmod → head: 3
13 de ADP case → head: 15
14 la DET det → head: 15
15 rue NOUN nmod → head: 12
16 de ADP case → head: 18
17 le DET det → head: 18
18 Petit-Lion PROPN nmod → head: 15
19 , PUNCT punct → head: 12
20 existait VERB root → head: 0
21 naguère ADV advmod → head: 20
22 une PRON nsubj → head: 20
23 de ADP case → head: 25
24 ces DET det → head: 25
25 maisons NOUN nmod → head: 22
26 précieuses ADJ amod → head: 25
27 qui PRON nsubj → head: 28
28 donnent VERB acl:relcl → head: 25
29 à ADP case → head: 31
30 les DET det → head: 31
31 historiens NOUN obl:arg → head: 28
32 la DET det → head: 33
33 facilité NOUN obj → head: 28
34 de ADP mark → head: 35
35 reconstruire VERB acl → head